In [ ]:
#| default_exp analyst_langraph

# Table-attached LangGraph analyst (Plotly)

Runs on **in-memory rows** from a Graph-tab table attached in chat.
Generates Plotly code via Lisette and returns a figure payload for `io_viz_view`.

Use `run_analyst(question, rows)` from the FastHTML shell.

In [ ]:
#| export
import json
import os
import re
from contextvars import ContextVar
from functools import lru_cache
from typing import Any, TypedDict

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from langgraph.graph import END, START, StateGraph
from lisette import Chat, contents

_current_df: ContextVar[pd.DataFrame | None] = ContextVar("analyst_df", default=None)


def content_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, list):
        return "".join(
            (x.get("text") or "") if isinstance(x, dict) else str(x) for x in content
        )
    return str(content)


def asdict(m: Any) -> dict:
    if isinstance(m, dict):
        return m
    if hasattr(m, "model_dump"):
        return m.model_dump()
    try:
        return dict(m)
    except Exception as ex:  # noqa: BLE001
        raise TypeError(f"cannot convert {type(m)} to dict") from ex


In [ ]:
#| export
class AnalystState(TypedDict, total=False):
    question: str
    columns: str
    result: str
    plot_code: str
    plotly: dict[str, Any] | None


def get_lisette_llm_config() -> dict[str, str]:
    model = (os.getenv("LISETTE_MODEL") or "").strip()
    key = (os.getenv("LISETTE_API_KEY") or "").strip() or (
        os.getenv("OPENAI_API_KEY") or ""
    ).strip()
    base = (os.getenv("LISETTE_API_BASE") or "").strip()
    if not model:
        return {}
    out: dict[str, str] = {"model": model}
    if key:
        out["api_key"] = key
    if base:
        out["api_base"] = base
    return out


def _active_df() -> pd.DataFrame:
    df = _current_df.get()
    if df is None:
        raise ValueError("No dataframe bound for analyst run")
    return df


In [ ]:
#| export
def analyze_data(state: AnalystState) -> dict:
    df = _active_df()
    cols = list(df.columns.astype(str))
    summary = (
        f"shape={df.shape[0]} rows × {df.shape[1]} cols\n"
        f"dtypes:\n{df.dtypes.to_string()}\n\n"
        f"head:\n{df.head(8).to_string()}"
    )
    result = f"Question: {state.get('question', '')}\n\nData summary:\n{summary}"
    return {"result": result, "columns": ",".join(cols)}


def llm_analyze(state: AnalystState) -> dict:
    prompt = (
        "You are a concise data analyst. Given the question and a small data summary, "
        "describe what chart would answer the question in 1-3 sentences. "
        "Do not write code.\n\n"
        f"Question: {state.get('question', '')}\n\n"
        f"Summary:\n{state.get('result', '')}"
    )
    cfg = get_lisette_llm_config()
    model = cfg.get("model")
    if not model:
        return {"result": state.get("result", "") + "\n\n(No LISETTE_MODEL; skipping analysis.)"}
    try:
        c = Chat(
            model,
            temp=0,
            api_key=cfg.get("api_key"),
            api_base=cfg.get("api_base"),
        )
        raw_res = c(prompt, max_steps=3, max_tokens=800, return_all=False, stream=False)
        final = contents(raw_res)
        res = content_text(asdict(final).get("content")) if final else ""
    except Exception as e:  # noqa: BLE001
        return {"result": f"{state.get('result', '')}\n\nLLM error: {e}"}
    return {"result": res or state.get("result", "")}


def generate_plot_code(state: AnalystState) -> dict:
    columns = [c for c in (state.get("columns") or "").split(",") if c]
    prompt = f"""You are a pandas + Plotly analyst. Write Python code to visualize the user's question.

Question: {state.get('question', '')}

The dataframe `df` is ALREADY loaded. Columns: {', '.join(columns)}

STRICT RULES:
1. `df` is already available. NEVER recreate or reload data.
2. `pd`, `px` (plotly.express), and `go` (plotly.graph_objects) are already imported. NEVER import libraries.
3. Assign the final Plotly figure to a variable named `fig`.
4. NEVER call `fig.show()` or `plt.show()`.
5. Return ONLY Python code — no markdown fences, no commentary.

Write the code now:"""

    cfg = get_lisette_llm_config()
    model = cfg.get("model")
    if not model:
        return {"plot_code": "# No LISETTE_MODEL configured"}
    try:
        c = Chat(
            model,
            temp=0,
            api_key=cfg.get("api_key"),
            api_base=cfg.get("api_base"),
        )
        raw_res = c(prompt, max_steps=3, max_tokens=2000, return_all=False, stream=False)
        final = contents(raw_res)
        text = content_text(asdict(final).get("content")) if final else ""
    except Exception as e:  # noqa: BLE001
        return {"plot_code": f"# LLM error: {e}"}

    code_match = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    plot_code = code_match.group(1).strip() if code_match else text.strip()
    return {"plot_code": plot_code}


def _jsonable(obj: Any) -> Any:
    """Convert Plotly/numpy objects into plain JSON-serializable values."""
    if obj is None or isinstance(obj, (str, bool, int, float)):
        return obj
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    # numpy scalars / arrays (and pandas NA)
    tolist = getattr(obj, "tolist", None)
    if callable(tolist):
        try:
            return _jsonable(tolist())
        except Exception:  # noqa: BLE001
            pass
    item = getattr(obj, "item", None)
    if callable(item):
        try:
            return _jsonable(item())
        except Exception:  # noqa: BLE001
            pass
    if hasattr(obj, "isoformat") and callable(obj.isoformat):
        try:
            return obj.isoformat()
        except Exception:  # noqa: BLE001
            pass
    return str(obj)


def _figure_to_plotly_payload(fig: Any) -> dict[str, Any] | None:
    if fig is None:
        return None
    payload: Any = None
    # Prefer Plotly's JSON string path — already numpy-safe when parsed.
    to_json_str = getattr(fig, "to_json", None)
    if callable(to_json_str):
        try:
            payload = json.loads(to_json_str())
        except Exception:  # noqa: BLE001
            payload = None
    if payload is None and isinstance(fig, dict) and "data" in fig:
        payload = fig
    if payload is None:
        to_json = getattr(fig, "to_plotly_json", None)
        if callable(to_json):
            raw = to_json()
            payload = json.loads(raw) if isinstance(raw, str) else raw
    if payload is None:
        to_dict = getattr(fig, "to_dict", None)
        if callable(to_dict):
            payload = to_dict()
    if not isinstance(payload, dict):
        return None
    clean = _jsonable(payload)
    if not isinstance(clean, dict):
        return None
    return {
        "data": clean.get("data") or [],
        "layout": clean.get("layout") or {},
    }


def execute_plot(state: AnalystState) -> dict:
    code = (state.get("plot_code") or "").strip()
    analysis = state.get("result", "") or ""
    if not code or code.startswith("#"):
        return {
            "result": analysis + "\n\nNo plot code generated.",
            "plotly": None,
        }

    df = _active_df()
    namespace: dict[str, Any] = {
        "df": df,
        "pd": pd,
        "px": px,
        "go": go,
        "fig": None,
    }
    try:
        exec(code, namespace)  # noqa: S102
        fig = namespace.get("fig")
        plotly_payload = _figure_to_plotly_payload(fig)
        if not plotly_payload or not plotly_payload.get("data"):
            return {
                "result": analysis + "\n\nPlot error: code did not assign a Plotly `fig`.",
                "plotly": None,
            }
        return {
            "result": analysis + "\n\nPlotly figure ready.",
            "plotly": plotly_payload,
        }
    except Exception as e:  # noqa: BLE001
        return {
            "result": analysis + f"\n\nPlot error: {e}",
            "plotly": None,
        }


In [ ]:
#| export
def build_graph():
    graph = StateGraph(AnalystState)
    graph.add_node("analyze", analyze_data)
    graph.add_node("llm_analyze", llm_analyze)
    graph.add_node("generate_plot_code", generate_plot_code)
    graph.add_node("execute_plot", execute_plot)
    graph.add_edge(START, "analyze")
    graph.add_edge("analyze", "llm_analyze")
    graph.add_edge("llm_analyze", "generate_plot_code")
    graph.add_edge("generate_plot_code", "execute_plot")
    graph.add_edge("execute_plot", END)
    return graph.compile()


@lru_cache(maxsize=1)
def get_graph():
    return build_graph()


def run_analyst(question: str, rows: list[dict[str, Any]]) -> dict[str, Any]:
    """Run the Plotly analyst on in-memory table rows for the chat UI."""
    if not rows:
        return {
            "status": "error",
            "error": "No table rows to analyze.",
            "response": "No table rows to analyze.",
            "visualization": "table_analyst",
            "endpoint": "table_analyst",
            "source": "analyst",
        }
    df = pd.DataFrame(rows)
    token = _current_df.set(df)
    try:
        out = get_graph().invoke(
            {
                "question": (question or "").strip(),
                "columns": ",".join(map(str, df.columns.tolist())),
                "result": "",
                "plot_code": "",
                "plotly": None,
            }
        )
    except Exception as e:  # noqa: BLE001
        return {
            "status": "error",
            "error": str(e),
            "response": f"Analyst failed: {e}",
            "visualization": "table_analyst",
            "endpoint": "table_analyst",
            "source": "analyst",
        }
    finally:
        _current_df.reset(token)

    plotly_payload = out.get("plotly") if isinstance(out, dict) else None
    response = (out.get("result") if isinstance(out, dict) else None) or ""
    if plotly_payload and plotly_payload.get("data"):
        return {
            "status": "success",
            "response": response,
            "plotly": plotly_payload,
            "visualization": "table_analyst",
            "endpoint": "table_analyst",
            "source": "analyst",
        }
    return {
        "status": "error",
        "error": "No Plotly figure produced.",
        "response": response or "No Plotly figure produced.",
        "visualization": "table_analyst",
        "endpoint": "table_analyst",
        "source": "analyst",
        "hint": "Try asking for a specific chart using the attached table columns.",
    }


In [ ]:
# Demo (not exported): requires LISETTE_* and sample rows
# rows = [{"mwt": 250.0, "logp": 1.2}, {"mwt": 300.0, "logp": 2.1}]
# run_analyst("Scatter of mwt vs logp", rows)